# The Halting Diagonal — Data

**No external datasets.** Every number in this paper is computed from a
definition. There is nothing to download, no observation to trust, and no
instrument to calibrate. That is the point of a CS paper: the data *is* the
execution.

Sources of every quantity:

| Quantity | Produced by | Arithmetic |
|---|---|---|
| Halting-table escape | construction, exhaustive enumeration | exact, integer |
| `D_n` derangement counts | `fractions.Fraction` + integer recurrence | exact, unbounded |
| `i^k`, `eₖ²` | `numpy` matrix powers; Cayley–Dickson product | exact on ±1 |
| Crib survival | direct enumeration of all alignments | exact counts |

The only floating point that enters a *claim* is `round(n!/e)` in P2 and the
2% tolerance in P4; both are stated as such in the predictions.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from ValaQuenta.modules.turing_diagonal import maths as td
from ValaQuenta import zero_lattice as zl

import math, itertools
from fractions import Fraction
import numpy as np

print('engine   : ValaQuenta.modules.turing_diagonal')
print('python   :', sys.version.split()[0])

## D1 — The engine's own report

In [ ]:
full = td.full_turing_diagonal()
print('theme:', full['theme'])
print()
for key in sorted(full):
    if key in ('theme',):
        continue
    val = full[key]
    if isinstance(val, dict) and 'claim' in val:
        print(f'{key}:')
        print(f'   {val["claim"]}')

## D2 — The halting table and the escaping row

The engine builds a table and constructs `D`. The check `D_in_table` is the
whole diagonal lemma, executed.

In [ ]:
h = td.turing_halting_diagonal(n_programs=50)
da = h['diagonal_argument']
print('n_programs :', h['n_programs'])
for k, v in da.items():
    print(f'  {k:>22} : {v}')
print()
print('D(D) contradiction:')
for k, v in h['D_of_D_contradiction'].items():
    print(f'  {k:>22} : {v}')

### D2b — Independent reconstruction

The engine uses one seeded table. The claim is about *every* table, so it is
re-derived here from scratch and checked exhaustively on the diagonal, where
the property actually lives.

In [ ]:
# Exhaustive over all 2^n diagonals. The off-diagonal entries cannot
# affect the escape, so this covers all 2^(n*n) tables of each order.
rows = []
for n in range(1, 17):
    ok = all(all((1 - d[i]) != d[i] for i in range(n))
             for d in itertools.product((0, 1), repeat=n))
    rows.append((n, 2 ** n, 2 ** (n * n), ok))

print(f'{"n":>3} {"diagonals":>10} {"tables":>22} {"escape holds":>13}')
for n, nd, nt, ok in rows:
    nt_s = f'{nt:.3e}' if nt > 10**9 else str(nt)
    print(f'{n:>3} {nd:>10} {nt_s:>22} {str(ok):>13}')
print()
print('The middle column is what exhaustive verification would have cost')
print('without the observation that only the diagonal matters.')

In [ ]:
# And on full random tables, D must not appear as any row.
rng = np.random.default_rng(0)
found = 0
for _ in range(2000):
    n = int(rng.integers(2, 12))
    T = rng.integers(0, 2, size=(n, n))
    D = 1 - np.diag(T)
    found += any(np.array_equal(D, T[i]) for i in range(n))
print(f'random tables tested : 2000  (order 2..11)')
print(f'times D was in table : {found}')

## D3 — Derangement counts, three routes

In [ ]:
def D_series(m):
    return int(sum(Fraction((-1) ** k, math.factorial(k))
                   for k in range(m + 1)) * math.factorial(m))

def D_recur(m):
    a, b = 1, 0
    if m == 0: return a
    if m == 1: return b
    for k in range(2, m + 1):
        a, b = b, (k - 1) * (b + a)
    return b

print(f'{"n":>3} {"D_n":>22} {"D_n/n!":>20} {"|D_n/n! - 1/e|":>16}')
for m in range(1, 15):
    Dm, fm = D_recur(m), math.factorial(m)
    print(f'{m:>3} {Dm:>22} {Dm/fm:>20.16f} {abs(Dm/fm - 1/math.e):>16.3e}')

print()
print('agreement, exact integers:')
print('  series == recurrence, n=0..60 :',
      all(D_series(m) == D_recur(m) for m in range(61)))
print('  recurrence == round(n!/e), n=1..17 :',
      all(D_recur(m) == round(math.factorial(m)/math.e) for m in range(1, 18)))

### D3b — The Enigma case, n = 26

26 letters, so the reflector is a derangement of 26 elements.

In [ ]:
e = td.enigma_derangement(26)
D26, f26 = e['D_n'], e['n_factorial']
print(f'D_26        = {D26}')
print(f'26!         = {f26}')
print(f'D_26 / 26!  = {D26/f26!r}')
print(f'1/e         = {1/math.e!r}')
print(f'difference  = {abs(D26/f26 - 1/math.e):.3e}')
print()
print('engine agrees with independent recurrence :', D26 == D_recur(26))
print()
print('At n=26 the alternating series has converged below float64 epsilon,')
print('so the derangement fraction and 1/e are the same double.')

## D4 — The involution at every level

In [ ]:
I2  = np.eye(2)
i_m = np.array([[0.0, -1.0], [1.0, 0.0]])
print('powers of i in the 2x2 real representation:')
for k in range(5):
    M = np.linalg.matrix_power(i_m, k)
    print(f'  i^{k} = [[{M[0,0]:+.0f},{M[0,1]:+.0f}],[{M[1,0]:+.0f},{M[1,1]:+.0f}]]')
print()
print(f'i^2 == -I  : {np.array_equal(np.linalg.matrix_power(i_m,2), -I2)}')
print(f'i^4 == +I  : {np.array_equal(np.linalg.matrix_power(i_m,4),  I2)}')
print(f'det(-I)    : {np.linalg.det(-I2):+.0f}   (orientation preserved)')
print(f'trace(-I)  : {np.trace(-I2):+.0f}')
print(f'eigenvalues: {np.linalg.eigvals(-I2)}   (no fixed vector but 0)')

In [ ]:
# Cayley-Dickson: square every sedenion basis element.
sq = [zl.multiply(zl.e_k(k), zl.e_k(k))[0] for k in range(16)]
print(f'{"k":>3} {"e_k^2":>8}')
for k in range(16):
    mark = '  <- the unique fixed point' if sq[k] > 0 else ''
    print(f'{k:>3} {sq[k]:>+8.0f}{mark}')
print()
neg = [k for k in range(16) if abs(sq[k] + 1) < 1e-12]
pos = [k for k in range(16) if abs(sq[k] - 1) < 1e-12]
print(f'e_k^2 = -1 for k = {neg}   (count {len(neg)})')
print(f'e_k^2 = +1 for k = {pos}')
print()
print('15 derangements, 1 fixed point. The fixed point is e_0 = 1,')
print('the multiplicative identity -- the Null Operator.')

## D5 — Crib pruning by the no-self-encryption constraint

The Enigma reflector's wiring made `cipher[k] = plain[k]` impossible. Turing's
Bombe exploited this: at any crib alignment where the ciphertext happens to
agree with the crib, that alignment is *ruled out before any rotor is turned*.

This measures how much of the search that removes.

In [ ]:
A, N = 26, 200_000
rng = np.random.default_rng(20260606)
cipher = rng.integers(0, A, size=N)

print(f'alphabet A = {A}, ciphertext length N = {N}')
print()
print(f'{"L":>4} {"alignments":>12} {"survived":>10} {"observed":>10} '
      f'{"(1-1/A)^L":>10} {"ratio":>8}')
crib_rows = []
for L in (4, 8, 12, 16, 20, 25):
    crib = rng.integers(0, A, size=L)
    n_align = N - L + 1
    surv = sum(1 for s in range(n_align) if not np.any(cipher[s:s+L] == crib))
    obs, pred = surv / n_align, (1 - 1/A) ** L
    crib_rows.append((L, n_align, surv, obs, pred))
    print(f'{L:>4} {n_align:>12} {surv:>10} {obs:>10.5f} {pred:>10.5f} '
          f'{obs/pred:>8.4f}')

print()
print('Every alignment discarded here costs one comparison, not a rotor search.')